This is a test implementation for the ID3 decision tree. If it works well, it will be moved to our implementations.py file. ID3 assumes all features are categorical, so we have to split numerical data into categories to make use of them.

In [1]:
# Imports.
import sys
from implementations import *
import numpy as np
from helpers import *
from typing import List
from math import log2
from ID3 import *

In [2]:
# Input the data

# set hyperparams
k = 2 # bin size
max_depth = 8 # max ID3 tree depth
#

path_to_dataset = "data/dataset"
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(
    path_to_dataset, max_rows=5000, max_features=40, NaNstrat="fill"
) # loading is too slow, I use max_rows and max_features for testing -M

# I'm skipping preprocessing as I don't need to standardize when I'm going to split the numerical data into bins either way.
# Ideally I still remove 0-variance columns.
mask = x_train.std(axis=0) != 0
x_train = x_train[:, mask]

# ID3 specific: convert the labels to strings
y_train = y_train[:].astype(str).reshape(-1, 1)
# Discretize numeric features
tx = x_train
bins = compute_bins(tx, k)
tx_disc = apply_bins(tx, bins)
x_test_disc = apply_bins(x_test, bins)
# slap x and y together to comply to ID3 method's format
train_data = np.hstack((tx_disc, y_train)) 
# we got rid of feature names in preprocess and I don't want to change that code so I assign some names here
dummy_header = np.array([f"col{i}" for i in range(tx_disc.shape[1])])
dummy_header_test = np.array([f"col{i}" for i in range(tx_disc.shape[1] - 1)])

dummy_column = np.full((x_test_disc.shape[0], 1), "x") # placeholder because ID3::predict expects a placeholder column
test_data = np.hstack([x_test_disc, dummy_column])



'\nprint(tx.shape)\nprint(len(bins))\nprint(x_test.shape)\nprint(tx_disc)\nprint(x_test_disc)\na = input()\n'

In [3]:
# Train the model.
def test_hyperparams(max_depth : int):
    for depth in range(1, max_depth+1):
        model = ID3()
        model.fit(dummy_header, train_data, depth, verbose=False)
        # print(model.tree.get_representation())
        # Run the model on test data
        predictions = model.predict(dummy_header_test, test_data, verbose=False)
        no_positives = 0
        # print(predictions)
        for pr in predictions:
            if pr == "1":
                no_positives += 1
        print(f"{no_positives} positive predictions")     
        # print(model.tree.get_representation())

In [4]:
test_hyperparams(max_depth)

0 positive predictions
0 positive predictions
0 positive predictions
3 positive predictions
3 positive predictions
105 positive predictions
105 positive predictions
105 positive predictions


In [ ]:
# TODO data split for cross validation and confusion matrix